In [1]:
import pandas as pd
from orderbook import OrderBook
import numpy as np
from scipy.stats import norm
from orderbook import unittest1

In [2]:
# read in bidding data
bidding_data = pd.read_csv("./data/players_L1.csv")
sell_offers = bidding_data[:24]["P_n"].copy()
buy_offers = bidding_data[24:]["P_n"].copy()
sim_data = pd.read_excel("./data/input_data.xlsx")
sim_data.set_index(["Time","ID"],inplace=True)
avg_buy_price = sim_data["buy_price"].mean()
avg_sell_price = sim_data["sell_price"].mean()


In [3]:
# fit norm. dists for buyer and seller offers
mu_s, std_s = norm.fit(sell_offers/100)
mu_b, std_b = norm.fit(buy_offers/100)

# drawing bids
sim_data["is_seller"]=sim_data.pv_gen > sim_data.load
sim_data["bid_price"]=sim_data.apply(lambda row: norm.rvs(mu_s-0.07,std_s) if row.is_seller else norm.rvs(mu_b-0.07,std_b),axis=1)

In [9]:
#add energy_sold 
sim_data["energy_in"] =np.zeros(len(sim_data))
sim_data["energy_out"] =np.zeros(len(sim_data))
sim_data["money_in"] =np.zeros(len(sim_data))
sim_data["money_out"] =np.zeros(len(sim_data))
sim_data["surplus"]=sim_data["pv_gen"]-sim_data["load"]

def execute_order(trader_buy,trader_sell,price,size,i=0):
    sim_data.loc[(i,trader_buy),"energy_in"] += size
    sim_data.loc[(i,trader_buy),"money_out"] += price/1000*size
    sim_data.loc[(i,trader_sell),"energy_out"] += size
    sim_data.loc[(i,trader_sell),"money_in"] += price/1000*size

for i in range(1,97):
    print(i)
    #slice = sim_data.xs(i).sample(n=250)
    slice = sim_data.xs(i).sort_values(by=["surplus"],ascending=False)
    ob = OrderBook("ob",cb=execute_order)
    for id,row in slice.iterrows():
        ob.limit_order(int(row.is_seller),abs(row.pv_gen-row.load),round(row.bid_price*1000),id,i)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96


In [10]:
sim_data.to_excel("./simulation_results/order_book_results.xlsx")

In [17]:
ob= OrderBook("x")
ob.limit_order(1,4,130,1)
ob.limit_order(0,2,130,1)

EXECUTE: 1 BUY 1 SELL 2 x @ 130


4

In [55]:
sim_data.xs((95,1))

load          0.245914
pv_gen             0.0
buy_price        0.171
sell_price       0.045
clock         23:30:00
Name: (95, 1), dtype: object